# Chẩn đoán TN3 — trong tệp nén trên Drive thật sự có gì

Notebook **chỉ đọc Drive**, không train, không ghi đè, không xoá gì trên Drive.
Chạy hết mất khoảng một phút.

## Vấn đề đang tìm

Ô khôi phục trong notebook TN3 c192 in ra `khôi phục 4 dòng`. Bốn mức alpha đã
chạy xong phải cho khoảng **20 dòng** — 4 alpha × (4 fold + 1 dòng TONG).

Hai khả năng, notebook này phân biệt chúng:

**A. Tệp nén thiếu.** Mỗi tệp chỉ chứa dòng của riêng nó. Mục 3 sẽ thấy mỗi
tệp 4–5 dòng.

**B. Phép gộp của tôi sai.** Ô khôi phục giải nén mọi tệp vào cùng `runs/`, nên
`runs/tn3/summary.csv` bị tệp sau đè tệp trước, chỉ còn một bản. Mục 3 sẽ thấy
tệp cuối có ~20 dòng mà kết quả gộp vẫn ra 4.

Mục 4 chạy cách gộp chắc chắn hơn — giải nén **từng tệp vào thư mục riêng** rồi
mới gộp — để so.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Có những tệp nén nào

In [ ]:
import glob, os, subprocess
DRIVE = "/content/drive/MyDrive/mobivital/"
for loc in ("c64", "c192"):
    z = sorted(glob.glob(DRIVE + "tn3_*%s*.zip" % loc))
    print("%-5s %d tệp" % (loc, len(z)))
    for f in z:
        gio = subprocess.run(["date","-r",f,"+%m-%d %H:%M"],capture_output=True,text=True).stdout.strip()
        print("   %s  %6.2f MB  %s" % (gio, os.path.getsize(f)/1e6, os.path.basename(f)[:64]))
    print()

## 3. Mỗi tệp nén chứa bao nhiêu dòng

Đây là ô phân biệt hai khả năng. Đọc `summary.csv` **bên trong** từng tệp, không
giải nén ra đĩa.

Nếu số dòng **tăng dần** theo thời gian thì tệp nén đủ, lỗi ở phép gộp.
Nếu tệp nào cũng chỉ 4–5 dòng thì tệp nén thiếu.

In [ ]:
for loc in ("c64", "c192"):
    print("===", loc)
    for f in sorted(glob.glob(DRIVE + "tn3_*%s*.zip" % loc)):
        r = subprocess.run(["unzip","-p",f,"tn3/summary.csv"], capture_output=True, text=True)
        n = len([l for l in r.stdout.splitlines()[1:] if l.strip()])
        print("   %2d dòng   %s" % (n, os.path.basename(f)[:66]))
    print()

## 4. Gộp theo cách chắc chắn hơn

Giải nén **từng tệp vào thư mục tạm riêng** rồi mới gộp, thay vì đè lên nhau.

In [ ]:
import csv, shutil, tempfile
def gop(loc):
    rows, seen = [], set()
    for f in sorted(glob.glob(DRIVE + "tn3_*%s*.zip" % loc)):
        d = tempfile.mkdtemp()
        subprocess.run(["unzip","-oq",f,"-d",d], check=True)
        for s in glob.glob(d + "/*/summary.csv"):
            for r in csv.DictReader(open(s)):
                k = (r.get("experiment"), r.get("run_id"))
                if k not in seen: seen.add(k); rows.append(r)
        shutil.rmtree(d)
    return rows

In [ ]:
KQ = {loc: gop(loc) for loc in ("c64", "c192")}
for loc, rows in KQ.items():
    print("%-5s gộp được %d dòng" % (loc, len(rows)))

## 5. Alpha nào đã xong

`·` là fold chưa chạy. `TONG` nghĩa là alpha đó đã đủ bốn fold.

In [ ]:
import re
from collections import defaultdict
FOLD = ["val_AB", "val_CE", "val_DF", "val_KL"]
for loc, rows in KQ.items():
    xong = defaultdict(set)
    for r in rows:
        m = re.search(r"_a([\d.]+)_corr.*?_seed\d+(?:_(val_\w+)|_tong)?$", r["run_id"])
        if m and ("_%s_" % loc) in r["run_id"]:
            xong[float(m.group(1))].add(m.group(2) or "TONG")
    print("===", loc)
    for a in sorted(xong):
        co = xong[a]
        print("   alpha %.1f  %s  %s" % (a, " ".join("%-7s" % (f if f in co else "·") for f in FOLD),
              "TONG" if "TONG" in co else ""))
    print()

## 6. Điểm của các alpha đã xong

In [ ]:
for loc, rows in KQ.items():
    print("===", loc)
    d = {}
    for r in rows:
        if r["fold"] != "TONG": continue
        m = re.search(r"_a([\d.]+)_", r["run_id"])
        if m and ("_%s_" % loc) in r["run_id"]: d[float(m.group(1))] = float(r["score_macro"])
    for a in sorted(d): print("   alpha %.1f   %.6f" % (a, d[a]))
    print("   (MSE thuần alpha 1,0: c64 0,760878 · c192 0,764428)\n")

## 7. Kết luận

Đọc mục 3 và mục 4 cùng nhau:

| mục 3 | mục 4 | nghĩa là |
|---|---|---|
| số dòng tăng dần, tệp cuối ~20 | ra ~20 dòng | tệp nén đủ, **phép gộp cũ sai** — dùng cách ở mục 4 |
| tệp nào cũng 4–5 dòng | vẫn ra ~20 dòng | tệp nén mỗi cái một phần, gộp lại là đủ |
| tệp nào cũng 4–5 dòng | ra 4–5 dòng | **tệp nén thật sự thiếu**, phải chạy lại |

Gửi tôi đầu ra mục 3 và mục 4, tôi vá notebook TN3 cho đúng.